# Chapter 14: evaluate the first 20 measures and print the test table

Section 14.2 loads the Kestrel gold tables, evaluates the 20 dictionary measures and prints each beside its
hand calculated check value. Each measure sums a numerator, sums a denominator and divides the totals; a zero
denominator returns blank. Entries the book sets "on the first load" show the value the slice gives.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import kestrel as k

con = k.connect()
kpi = k.kpi_tables(con)
months = k.month_balance()
march = months.set_index("month").loc["2026-03"]
dictionary = pd.read_csv(k.DATA / "kpi_dictionary.csv")

def ratio(df, name, scale=100, **where):
    rows = df[df.kpi_name == name]
    for col, val in where.items():
        rows = rows[rows[col] == val]
    den = rows.denominator.sum()
    return None if den == 0 else scale * rows.numerator.sum() / den

def key_of(table, id_col, natural):
    return k.q(con, f"SELECT * FROM {table} WHERE {id_col} = ?", (natural,)).iloc[0, 0]

brew68 = key_of("brew", "brew_id", "B26-0068")
f07 = key_of("batch", "batch_id", "F07-26-034")

In [ ]:
values = {}
values[1] = ratio(kpi["kpi_brew"], "Extract efficiency", brew_key=brew68)
values[2] = ratio(kpi["kpi_brew"], "Wort volume yield", 1, brew_key=brew68)
values[3] = ratio(kpi["kpi_brew"], "Cast gravity deviation", 1, brew_key=brew68)
values[4] = 100 * (months.cast_hl.sum() / months.brews.sum() - 300) / 300
values[5] = ratio(kpi["kpi_fill"], "Apparent attenuation", batch_key=f07)
values[6] = 100 * march.cellar_loss_hl / march.cast_hl
values[7] = 100 * march.filtration_loss_hl / march.cast_hl
fermenters = k.q(con, "SELECT COUNT(*) AS n FROM vessel WHERE vessel_type = 'fermenter' AND valid_to IS NULL").n[0]
values[8] = 100 * months.tank_days_used.sum() / (fermenters * 365)
cost = k.load("stage_cost").set_index("stage").carried_cost_usd_per_hl
end_ferm = k.q(con, "SELECT volume_hl FROM vessel_event WHERE event_type = 'fermentation end' AND batch_key = ?",
               (int(f07),)).volume_hl[0]
values[9] = end_ferm * cost["end of fermentation"]
values[10] = 100 * march.closure_gap_hl / march.cast_hl
ls = kpi["kpi_line_shift"]
values[11] = 100 * (ratio(ls, "OEE availability", 1) * ratio(ls, "OEE performance", 1) * ratio(ls, "OEE quality", 1))
values[12] = ratio(ls, "Line speed loss", equipment_key=2, shift_date="2026-04-13")
packed = march.cast_hl - march.cellar_loss_hl - march.filtration_loss_hl - march.packaging_loss_hl
values[13] = 100 * march.packaging_loss_hl / packed
values[14] = ratio(kpi["kpi_pack_lot"], "Reject rate")
cans = k.q(con, "SELECT lot_key FROM lot WHERE lot_type = 'pack' AND lot_id LIKE '%-C1-%'").lot_key
pl = kpi["kpi_pack_lot"]
values[15] = ratio(pl[pl.pack_lot_key.isin(cans)], "Fill giveaway", 1)

Quality entries. Right first time takes the first result of each test on each batch; spec compliance takes the
current results. Hold rate needs a hold record: the batch table of Appendix C has no hold column, so it returns
blank until one is added. Mock recall time comes from the drill log, which QA fills in.

In [ ]:
res = k.q(con, """
    SELECT s.batch_key, r.test_code, r.in_spec_flag, r.is_current, r.entered_at_utc
    FROM result r JOIN sample s USING (sample_key) WHERE r.spec_key IS NOT NULL""")
first = res.sort_values("entered_at_utc").groupby(["batch_key", "test_code"]).first().reset_index()
rft = first.groupby("batch_key").in_spec_flag.min()
values[16] = 100 * rft.sum() / len(rft)
values[17] = None
cur = res[res.is_current == 1]
values[18] = 100 * cur.in_spec_flag.sum() / len(cur)
complaints = k.load("bronze_qa_complaint")
shipped = k.q(con, """
    SELECT SUM(p.unit_count) AS packs FROM movement m JOIN package_unit p ON p.package_unit_key = m.from_node_key
    WHERE m.movement_type = 'ship' AND m.posted_utc LIKE '2026-04%'""").packs[0]
values[19] = (complaints.received_date.str[:7] == "2026-04").sum() / shipped * 1_000_000
drills = k.load("drill_log")
values[20] = drills.minutes_to_signed_list.mean() if len(drills) else None

In [ ]:
HAND = {1: 92.6, 2: 5.66, 3: 0.0, 4: -0.48, 5: 81.7, 6: 2.40, 7: 0.6, 8: 85.1, 9: 16200, 10: 0.25, 12: 20,
        13: 1.04, 15: 4}
DECIMALS = {1: 1, 2: 2, 3: 1, 4: 2, 5: 1, 6: 2, 7: 1, 8: 1, 9: 0, 10: 2, 12: 0, 13: 2, 15: 0}
rows = []
for i, entry in dictionary.iterrows():
    n = i + 1
    v = values[n]
    shown = None if v is None else round(v, DECIMALS.get(n, 2))
    test = "no check value yet" if n not in HAND else ("pass" if shown == HAND[n] else "FAIL")
    if v is None:
        test = "blank: " + ("no hold column in batch" if n == 17 else "drill log empty")
    rows.append((n, entry["name"], entry.grain, HAND.get(n), shown, test))
table = pd.DataFrame(rows, columns=["#", "measure", "grain", "hand value", "model value", "test"])
print(table.to_string(index=False))
assert not (table.test == "FAIL").any()